## Импорты

In [ ]:
try:
    import google.colab
    %pip install pandas torch transformers datasets evaluate sacrebleu bert-score sentence-transformers rapidfuzz deep-translator tqdm numpy matplotlib PyYAML
    %pip install --upgrade tiktoken protobuf sentencepiece
    %pip install accelerate>=1.1.0
    is_colab = True
except:
    is_colab = False

if is_colab:
    !git clone https://github.com/AbonentVneSeti/text_processing_final_project
    %cd text_processing_final_project

In [2]:
import sys
sys.path.append('..')
from utils import load_config, build_dataloaders
import pandas as pd
from models.trainer import train_model
from models.rut5_paraphraser.model import ParaphraserModel

config = load_config("config.yaml")

d:\programming projects\3 course\text_processing_final_project\.conda\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Загрузка датасета

In [3]:
import os
if not os.path.exists("data/paraphrases_clean.parquet"):
    !wget -O data/paraphrases_clean.parquet https://github.com/AbonentVneSeti/text_processing_final_project/main/data/paraphrases_clean.parquet

df = pd.read_parquet("data/paraphrases_clean.parquet")
print(f"Загружено {len(df)} пар")

Загружено 25 пар


In [4]:
train_loader, val_loader = build_dataloaders(df, config["model_config"])

## Обучение модели

In [5]:
# Ячейка 6: создание и обучение модели
model = ParaphraserModel(config["model_config"])
trained_model = train_model(
    model, train_loader, val_loader,
    config["model_config"], config["trainer_config"], config["metrics_config"]
)
trained_model.save(config["trainer_config"]["output_dir"])

Loading weights: 100%|██████████| 282/282 [00:00<00:00, 8108.60it/s]
[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.
Map: 100%|██████████| 3/3 [00:00<?, ? examples/s]
d:\programming projects\3 course\text_processing_final_project\.conda\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss
1,27.275227,14.565956
2,25.019484,15.695302
3,25.490095,17.472485
4,27.612774,18.388121
5,26.720135,17.917759


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.43it/s]
d:\programming projects\3 course\text_processing_final_project\.conda\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.47it/s]
d:\programming projects\3 course\text_processing_final_project\.conda\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.32it/s]
d:\programming projects\3 course\text_processing_final_project\.conda\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().

Training completed. Model saved to models/rut5_paraphraser/saves


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.07it/s]


## Сохранение (для Colab)

In [6]:
if is_colab:
    from google.colab import drive
    drive.mount('/content/drive')
    !cp -r models/rut5_paraphraser/saves /content/drive/MyDrive/
    print("Модель сохранена на Google Drive")